In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (TimeoutException, NoSuchElementException, 
                                      ElementClickInterceptedException, WebDriverException)
import time
import pandas as pd
import random
import re
import os
from typing import List, Dict

class Config:
    MAX_PROPERTIES = 7352
    WAIT_TIME = 20
    DELAY_RANGE = (3, 7)
    OUTPUT_FILE = "file12.csv"
    USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
    MAX_RETRIES = 5
    BATCH_SIZE = 100

In [ ]:

def iniciar_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--start-maximized")
    options.add_argument(f"user-agent={Config.USER_AGENT}")
    options.add_argument("--log-level=3")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    return uc.Chrome(options=options)

In [ ]:
def espera_aleatoria(min_max: tuple = Config.DELAY_RANGE) -> None:
    time.sleep(random.uniform(*min_max))

In [ ]:
def limpiar_texto(texto: str) -> str:
    if not texto:
        return "N/A"
    texto = re.sub(r'[^\w\s.,-]', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()

In [ ]:
def extraer_detalles(anuncio) -> Dict[str, str]:
    """Extrae y estructura los detalles de la propiedad."""
    detalles_dict = {
        "Recámaras": "N/A",
        "Baños": "N/A",
        "Parking": "N/A",
        "M² de construcción": "N/A"
    }
    try:
        detalles = anuncio.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__details-item")
        detalles_texto = [detalle.text.strip() for detalle in detalles]
        
        if len(detalles_texto) >= 3:
            detalles_dict["M² de construcción"] = detalles_texto[0].replace("\n", "")
            detalles_dict["Recámaras"] = detalles_texto[1]
            detalles_dict["Baños"] = detalles_texto[2].replace("Más", "N/A")
            if len(detalles_texto) > 3:
                detalles_dict["Parking"] = detalles_texto[3]
                
    except Exception as e:
        print(f" Error extrayendo detalles: {str(e)}")
    
    return detalles_dict

In [ ]:
def procesar_anuncio(anuncio) -> Dict[str, str]:
    try:
        precio_texto = anuncio.find_element(By.CSS_SELECTOR, ".d3-ad-tile__price").text.strip()
        precio_texto = re.sub(r'\n-\d+%', '', precio_texto).strip()
        precio_texto = precio_texto.replace("B.", "B/.")  # Formatear correctamente el precio
        
        ubicacion = anuncio.find_element(By.CSS_SELECTOR, ".d3-ad-tile__location").text.strip()
        
        detalles = extraer_detalles(anuncio)
        
        return {
            "Precio": limpiar_texto(precio_texto),
            "Ubicación": limpiar_texto(ubicacion),
            **detalles
        }
    except Exception as e:
        print(f" Error procesando anuncio: {str(e)}")
        return None


In [ ]:
def ir_a_siguiente_pagina(driver, page_number: int) -> bool:
    try:
        contenedor_paginacion = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".d3-category-list__bottom-pagination"))
        )
        
        siguiente_pagina = contenedor_paginacion.find_element(
            By.CSS_SELECTOR, ".d3-pagination__arrow.d3-pagination__arrow--next"
        )
        
        if siguiente_pagina.is_displayed() and siguiente_pagina.is_enabled():
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", siguiente_pagina)
            espera_aleatoria((1, 2))
            
            try:
                siguiente_pagina.click()
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", siguiente_pagina)
            
            WebDriverWait(driver, 15).until(EC.staleness_of(siguiente_pagina))
            
            print(f"Cargando página {page_number + 1}...")
            espera_aleatoria((3, 5))
            return True
        else:
            print(" El botón 'Siguiente página' no está disponible o habilitado")
            return False
            
    except TimeoutException:
        print(" Tiempo de espera agotado buscando paginación")
        return False
    except NoSuchElementException:
        print(" No se encontró el elemento de paginación")
        return False
    except ElementClickInterceptedException as e:
        print(f" Error al hacer clic (elemento interceptado): {e}")
        return False
    except Exception as e:
        print(f"Error : {e}")
        return False


In [ ]:
def extraer_propiedades_paginadas(driver, url: str, max_propiedades: int = Config.MAX_PROPERTIES) -> List[Dict]:
    driver.get(url)
    print(" Cargando ")
    propiedades = []
    page_number = 1
    fallos_consecutivos = 0
    max_fallos_permitidos = 2
    
    try:
        while len(propiedades) < max_propiedades and fallos_consecutivos < max_fallos_permitidos:
            print(f"\n Procesando  {page_number}...")
            
            try:
                WebDriverWait(driver, Config.WAIT_TIME).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".d3-ad-tile__content"))
                )
                fallos_consecutivos = 0
            except TimeoutException:
                fallos_consecutivos += 1
                print(f" Fallo consecutivo {fallos_consecutivos} al cargar anuncios")
                if fallos_consecutivos >= max_fallos_permitidos:
                    break
                driver.refresh()
                continue
            
            anuncios = driver.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__content")
            
            for anuncio in anuncios:
                if len(propiedades) >= max_propiedades:
                    break
                propiedad = procesar_anuncio(anuncio)
                if propiedad:
                    propiedades.append(propiedad)
                    print(f"🔹 Propiedad {len(propiedades)}: {propiedad['Precio']} | {propiedad['Ubicación']} | {propiedad['Recámaras']} rec | {propiedad['Baños']} baños | {propiedad['Parking']} est | {propiedad['M² de construcción']}")
            
            print(f"Página {page_number}: {len(propiedades)}/{max_propiedades} propiedades")
            
            if len(propiedades) >= max_propiedades:
                break
                
            if not ir_a_siguiente_pagina(driver, page_number):
                fallos_consecutivos += 1
                if fallos_consecutivos >= max_fallos_permitidos:
                    break
                print("Recargando página actual...")
                driver.get(driver.current_url)
                espera_aleatoria()
                continue
            
            page_number += 1
            fallos_consecutivos = 0
            espera_aleatoria()
            
    except Exception as e:
        print(f"Error crítico: {str(e)}")
    
    return propiedades[:max_propiedades]

In [ ]:
def guardar_datos_csv(propiedades: List[Dict], nombre_archivo: str = Config.OUTPUT_FILE) -> None:
    if not propiedades:
        print(" No hay datos para guardar")
        return
    
    try:
        columnas_deseadas = ['Precio', 'Ubicación', 'Recámaras', 'Baños', 'Parking', 'M² de construcción']
        df = pd.DataFrame(propiedades)[columnas_deseadas]
        
        if os.path.exists(nombre_archivo):
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            nombre_archivo = f"propiedades_encuentra24_{timestamp}.csv"
            print(f"Archivo existente detectado. Guardando como {nombre_archivo}")
        
        df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
        print(f" Datos guardados exitosamente en '{nombre_archivo}'")
        
        print("\n Resumen de datos guardados:")
        print(df.head())
        
    except Exception as e:
        print(f" Error al guardar los datos: {str(e)}")
        try:
            backup_file = "backup_" + nombre_archivo
            pd.DataFrame(propiedades)[columnas_deseadas].to_csv(backup_file, index=False, encoding='utf-8-sig')
            print(f"Se guardó una copia de respaldo en '{backup_file}'")
        except:
            print(" Error crítico: No se pudo guardar ningún archivo")

In [ ]:
def main():
    url = "https://www.encuentra24.com/panama-es/bienes-raices-venta-de-propiedades-casas"
    
    driver = None
    try:
        driver = iniciar_driver()
        print("Iniciando extracción de datos... (Solo 50 propiedades)")
        propiedades = extraer_propiedades_paginadas(driver, url)
        
        if propiedades:
            print("\nExtracción completada con éxito!")
            print(f"Total de propiedades obtenidas: {len(propiedades)}")
            guardar_datos_csv(propiedades)
        else:
            print("No se encontraron propiedades para guardar")
            
    except Exception as e:
        print(f"Error crítico en la ejecución: {str(e)}")
    finally:
        if driver:
            try:
                driver.quit()
                print(" Navegador cerrado correctamente")
            except WebDriverException as e:
                print(f" Error al cerrar el navegador: {str(e)}")

if __name__ == "__main__":
    main()